# Experiment: perplexity features
Open-weight multilingual LM (Qwen2.5-0.5B) scores each text (head+tail, 512 tokens, same as the encoder). Per-token log-prob, entropy and top-1 hits are summarized per text and per quarter of the text (fusion keeps the human beginning).
Classifier: HistGradientBoosting on the same 40k train sample as the encoder. Reference: TF-IDF baseline and XLM-R head+tail on the same val fold.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

warnings.filterwarnings('ignore')
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

SEED = 42
LM = 'Qwen/Qwen2.5-0.5B'
TRAIN_N = 40_000
MAX_LEN = 512
HEAD = 128
BATCH = 4
LABELS = ['human', 'ai', 'fusion']
INPUT = Path('/kaggle/input')

train = pd.read_parquet(next(INPUT.rglob('train_clean.parquet')), columns=['id', 'text', 'lang', 'domain', 'label', 'fold'])
test = pd.read_parquet(next(INPUT.rglob('test_clean.parquet')))
tr = train[train.fold == 'train']
tr = tr.groupby(['label', 'lang', 'domain'], group_keys=False).sample(frac=TRAIN_N / len(tr), random_state=SEED)
va = train[train.fold == 'val']
del train

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(LM)
tok.model_max_length = 10**6
pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
lm = AutoModelForCausalLM.from_pretrained(LM, dtype=torch.float16 if device == 'cuda' else torch.float32).to(device).eval()
len(tr), len(va), len(test)

(40001, 54537, 25000)

## Token-level scores

In [2]:
def encode(texts):
    out = []
    for x in tok(list(texts), add_special_tokens=False)['input_ids']:
        if len(x) > MAX_LEN:
            x = x[:HEAD] + x[len(x) - (MAX_LEN - HEAD):]
        out.append(x)
    return out

def summarize(lp, ent, top1):
    q = [x.mean() for x in np.array_split(lp, 4)]
    return {
        'lp_mean': lp.mean(), 'lp_std': lp.std(),
        **{f'lp_p{p}': np.percentile(lp, p) for p in [10, 25, 50, 90]},
        'ent_mean': ent.mean(), 'ent_std': ent.std(),
        'gap_mean': (lp + ent).mean(),
        'top1_rate': top1.mean(),
        **{f'lp_q{i}': x for i, x in enumerate(q)},
        'lp_q_std': np.std(q),
        'lp_tail_minus_head': np.mean(q[2:]) - np.mean(q[:2]),
        'log_tokens': np.log1p(len(lp)),
    }

@torch.no_grad()
def score(texts):
    ids = encode(texts)
    order = np.argsort([len(x) for x in ids])
    feats = [None] * len(ids)
    for b in range(0, len(order), BATCH):
        idx = order[b:b + BATCH]
        L = max(len(ids[i]) for i in idx)
        x = torch.full((len(idx), L), pad_id)
        m = torch.zeros((len(idx), L), dtype=torch.long)
        for r, i in enumerate(idx):
            x[r, :len(ids[i])] = torch.tensor(ids[i])
            m[r, :len(ids[i])] = 1
        x, m = x.to(device), m.to(device)
        logits = lm(input_ids=x, attention_mask=m).logits[:, :-1].float()
        logp = torch.log_softmax(logits, -1)
        tgt = x[:, 1:]
        lp = logp.gather(-1, tgt[..., None]).squeeze(-1)
        ent = -(logp.exp() * logp).sum(-1)
        top1 = (logits.argmax(-1) == tgt).float()
        valid = m[:, 1:].bool()
        for r, i in enumerate(idx):
            v = valid[r]
            if v.sum() < 2:
                v = torch.ones_like(v)
            feats[i] = summarize(lp[r][v].cpu().numpy(), ent[r][v].cpu().numpy(), top1[r][v].cpu().numpy())
        del logits, logp
    return pd.DataFrame(feats)

P = {}
for name, df in [('train', tr), ('val', va), ('test', test)]:
    P[name] = score(df.text).set_axis(df.id.values)
    P[name].rename_axis('id').to_csv(f'ppl_feats_{name}.csv')
    print(name, P[name].shape)

train (40001, 17)
val (54537, 17)
test (25000, 17)


In [3]:
P['val'].groupby(va.label.values).median().T.round(3)

,ai,fusion,human
lp_mean,-2.220,-2.840,-2.969
lp_std,2.436,2.769,2.836
lp_p10,-5.603,-6.729,-6.952
lp_p25,-3.386,-4.303,-4.513
lp_p50,-1.400,-2.050,-2.193
lp_p90,-0.034,-0.090,-0.099
ent_mean,2.329,2.821,2.931
ent_std,1.542,1.573,1.583
gap_mean,0.098,-0.024,-0.046
top1_rate,0.507,0.427,0.413


## Classifier on perplexity features

In [4]:
hgb = HistGradientBoostingClassifier(max_iter=400, learning_rate=.1, early_stopping=True, random_state=SEED)
hgb.fit(P['train'], tr.label)
for name, df in [('val', va), ('test', test)]:
    pd.DataFrame(np.log(hgb.predict_proba(P[name]) + 1e-9), columns=hgb.classes_, index=df.id).to_csv(f'{name}_probs_ppl.csv')

preds = {'perplexity': pd.Series(hgb.predict(P['val']), index=va.id)}
for f in list(INPUT.rglob('val_scores_baseline.csv')) + list(INPUT.rglob('val_logits_xlmr_headtail.csv')):
    preds[f.stem[4:]] = pd.read_csv(f, index_col='id').loc[va.id][LABELS].idxmax(axis=1)

y = va.set_index('id').label
def scores(p):
    f = f1_score(y, p.loc[y.index], labels=LABELS, average=None)
    return {**dict(zip(LABELS, f)), 'macro': f.mean()}

pd.DataFrame({k: scores(p) for k, p in preds.items()}).T.round(4)

,human,ai,fusion,macro
perplexity,0.5381,0.8526,0.5447,0.6451
scores_baseline,0.8425,0.9888,0.8493,0.8935
logits_xlmr_headtail,0.8431,0.9875,0.8740,0.9015


In [5]:
v = va.set_index('id')
pd.concat({by: v.assign(pred=preds['perplexity'].loc[v.index]).groupby(by).apply(lambda d: f1_score(d.label, d.pred, average='macro'))
           for by in ['lang', 'domain']}).round(4).rename('macro_f1').to_frame()

macro_f1
lang   de-DE                     0.6473
       en-EN                     0.6443
domain arxiv_papers              0.5382
       blog_authorship_corpus    0.5391
       bundestag                 0.6476
       cnn_news                  0.6619
       euro_court_cases          0.4398
       gutenberg                 0.5078
       house_of_commons          0.6593
       student_essays            0.7614